# 07 · Casos de decisión del Clausura 2026

**Qué hace.** Tres casos del torneo que el análisis no vio. Cada uno se elige con una regla escrita **antes** de ver lo que pasó (`contexto/plan_cierre.md`, fase B):
1. **Ganando por 1:** el partido del CL26 con más minutos ganando por un gol. ¿Se repitió lo que marcaba la matriz?
2. **Cambio de sistema:** el primer cambio de sistema propio del CL26. ¿Bajó el xG como anticipaba la alerta?
3. **La eliminatoria:** los cuartos de final. ¿Qué decía nuestro motor del rival, con solo partidos anteriores, y qué pasó?

De cada caso: qué marcaba la herramienta, qué pasó y una lectura. **Nunca se dice que la herramienta habría cambiado una decisión.**

**Qué necesita.** Pasos 02–05, el notebook 05 (`matriz.parquet`) y `scripts/08_maqueta.py` (las fichas en `outputs/app/datos.js`).

**Cómo probarlo.** *Run All*. Tarda menos de un minuto. Escribe `outputs/hallazgos/casos_cl26.json` (local), que lee `08_maqueta.py`.

In [ ]:
import json

import numpy as np
import pandas as pd

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import contexto, quiebres
from ojo.framework import partidos as fp
from ojo.lentes import indice, minuto, rivales, situaciones

P = params()
EQUIPO = P["equipo"]["team_id"]
DIR_HALLAZGOS = DIR_SALIDAS / "hallazgos"
BAJO, ALTO = P["indice"]["bandas"]

todos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
todos["prueba"] = fp.es_prueba(todos)
cl26 = todos[todos["prueba"]].copy()
ids = cl26["match_id"].astype(int).tolist()
eventos = almacen.cargar("eventos", ids, columnas=[
    "match_id", "indice", "periodo", "minuto", "equipo_id", "tipo", "tiro_resultado", "tarjeta", "formacion",
    "sustitucion_resultado", "x", "y", "fin_x", "fin_y", "duracion", "es_transicion", "directness"])
panel_todo = minuto.cargar_panel("minuto")
panel = panel_todo[panel_todo["match_id"].isin(ids)]

# Índice por minuto con la escala de los minutos de entrenamiento de Jardine, Solari y Ortiz (sin el CL26)
entrenamiento_ids = todos.loc[~todos["prueba"] & todos["entrenador_id"].isin(
    [d["manager_id"] for d in [*P["predecesores"], P["entrenador"]]]), "match_id"]
ev_entr = almacen.cargar("eventos", entrenamiento_ids.astype(int).tolist(), columnas=[
    "match_id", "periodo", "minuto", "equipo_id", "tipo", "duracion", "es_transicion", "directness", "x", "y", "fin_x", "fin_y"])
comp_entr = indice.componentes(panel_todo[panel_todo["match_id"].isin(entrenamiento_ids)], indice.piezas_por_minuto(ev_entr))
escala = indice.ajustar(comp_entr)
tab = indice.calcular(indice.componentes(panel, indice.piezas_por_minuto(eventos)), escala)
ind = tab.pivot_table(index=["match_id", "periodo", "minuto"], columns="lado", values="indice")
matriz = pd.read_parquet(DIR_HALLAZGOS / "matriz.parquet").set_index("situacion")
base = pd.read_parquet(DIR_HALLAZGOS / "matriz_base.parquet").iloc[0]
fichas = {f["rival"]: f for f in json.loads((DIR_SALIDAS / "app" / "datos.js").read_text(encoding="utf-8").split("=", 1)[1].rstrip().rstrip(";"))["decidir"]["fichas"]}


def titulo_partido(r: pd.Series) -> str:
    return f"América {int(r['goles_favor'])}-{int(r['goles_contra'])} {r['rival']} ({pd.Timestamp(r['fecha']):%d/%m/%Y})"


def goles(match_id: int) -> list[str]:
    ev = eventos[eventos["match_id"] == match_id].sort_values("indice")
    anota = contexto.goles_por_evento(ev)
    return [f"{int(m)}' {'América' if a == EQUIPO else 'rival'}" for m, a in zip(ev.loc[anota.notna(), "minuto"], anota.dropna())]


casos = []

## Caso 1 · Ganando por 1

In [ ]:
est = situaciones.por_minuto(eventos, EQUIPO)
est["situacion"] = situaciones.asignar(est)
mins = est[est["situacion"] == "gana_1"].groupby("match_id").size()
mid = int(mins.idxmax())
r = cl26.set_index("match_id").loc[mid]
claves = est[(est["match_id"] == mid) & (est["situacion"] == "gana_1")][["match_id", "periodo", "minuto"]]
m = claves.merge(ind.reset_index(), on=["match_id", "periodo", "minuto"], how="left")
xg = claves.merge(panel, on=["match_id", "periodo", "minuto"]).groupby("lado")["xg"].sum()
fila = matriz.loc["gana_1"]
ia, ir = m[minuto.PROPIO].mean(), m[minuto.RIVAL].mean()
repliegue = base["indice"] - ia >= P["matriz"]["dif_indice"]
caso = {
    "id": "ganando_por_1", "titulo": titulo_partido(r) + ": ganando por 1",
    "regla": "El partido del CL26 con más minutos ganando por un gol.",
    "marcaba": (f"Matriz de AP23–AP25: ganando por 1 el América se repliega (índice {fila['indice']:.0f} frente a {base['indice']:.0f} "
                f"de su partido normal) y el partido se empareja (balance de ocasiones {fila['xg_dif_90']:+.2f} xG cada 90 min, rango probable {fila['xg_dif_bajo']:+.2f} a "
                f"{fila['xg_dif_alto']:+.2f}). Hipótesis: sostener la iniciativa (no está probado que el repliegue cause el empate)."),
    "paso": [f"Ganó por un gol durante {int(mins.max())} minutos. Goles: {', '.join(goles(mid))}.",
             f"Índice de iniciativa del América en esos minutos: {ia:.0f} (rival {ir:.0f}).",
             f"Goles esperados (xG) en esos minutos: {xg.get(minuto.PROPIO, 0):.2f} a favor y {xg.get(minuto.RIVAL, 0):.2f} en contra.",
             f"Final: {int(r['goles_favor'])}-{int(r['goles_contra'])}."],
    "lectura": ("Se repitió el repliegue que marcaba la matriz" if repliegue else "Esta vez no se replegó como marcaba la matriz")
               + (" y el rival generó más que el América en esos minutos." if xg.get(minuto.RIVAL, 0) > xg.get(minuto.PROPIO, 0)
                  else " y aun así el América generó más que el rival en esos minutos."),
}
casos.append(caso)
caso

## Caso 2 · El primer cambio de sistema

In [ ]:
limites = quiebres.limites_periodo(eventos)
q = quiebres.ventanas(quiebres.detectar(eventos, EQUIPO), limites)
cs = q[(q["tipo"] == "cambio_sistema_propio") & q["valida"]].merge(cl26[["match_id", "fecha"]], on="match_id").sort_values(["fecha", "periodo", "minuto"]).iloc[0]
r = cl26.set_index("match_id").loc[int(cs["match_id"])]


def ventana(fase: str) -> pd.Series:
    p = panel[(panel["match_id"] == cs["match_id"]) & (panel["periodo"] == cs["periodo"])
              & panel["minuto"].between(cs[f"{fase}_desde"], cs[f"{fase}_hasta"])]
    n = cs[f"{fase}_hasta"] - cs[f"{fase}_desde"] + 1
    return 10 * p.groupby("lado")["xg"].sum() / n


antes, despues = ventana("antes"), ventana("despues")
caso = {
    "id": "cambio_de_sistema", "titulo": titulo_partido(r) + f": cambio de sistema al {int(cs['minuto'])}'",
    "regla": "El primer cambio de sistema propio del CL26 con ventanas válidas.",
    "marcaba": ("Alerta de AP23–AP25 (pasa los 4 filtros): tras sus cambios de sistema, los goles esperados (xG) del América bajan en los 10 minutos siguientes "
                "frente a lo que pasaba con Solari y Ortiz. Para el cuerpo técnico (hipótesis): cuidar el paso al nuevo sistema."),
    "paso": [f"Cambio de {cs['detalle']} al minuto {int(cs['minuto'])}.",
             f"xG del América cada 10 minutos: {antes.get(minuto.PROPIO, 0):.2f} antes y {despues.get(minuto.PROPIO, 0):.2f} después.",
             f"xG del rival cada 10 minutos: {antes.get(minuto.RIVAL, 0):.2f} antes y {despues.get(minuto.RIVAL, 0):.2f} después.",
             f"Final: {int(r['goles_favor'])}-{int(r['goles_contra'])}. Goles: {', '.join(goles(int(cs['match_id']))) or 'ninguno'}."],
    "lectura": (("El xG del América bajó después del cambio, como anticipaba la alerta"
                 if despues.get(minuto.PROPIO, 0) < antes.get(minuto.PROPIO, 0)
                 else "El xG del América no bajó después del cambio: esta vez la alerta no se cumplió")
                + ("; pero el del rival bajó más: el cambio frenó al rival."
                   if antes.get(minuto.RIVAL, 0) - despues.get(minuto.RIVAL, 0) > antes.get(minuto.PROPIO, 0) - despues.get(minuto.PROPIO, 0)
                   else ".")),
}
casos.append(caso)
caso

## Caso 3 · La eliminatoria

Lo que dice el motor del rival sale de sus partidos de AP23–AP25, todos anteriores a la serie. En la serie se comparan sus mismos rasgos (Team Match Stats) con su perfil.

In [ ]:
serie = cl26[cl26["etapa"].str.contains("Quarter", na=False)].sort_values("fecha")
nombre = serie["rival"].iloc[0]
rival_id = int(serie["rival_id"].iloc[0])
ficha = fichas.get(nombre)
liga = fp.con_torneo(fp.validos(almacen.cargar("partidos")))
liga["prueba"] = fp.es_prueba(liga)
ventana_ids = liga.loc[liga["torneo_corto"].isin(set(todos.loc[(todos["entrenador_id"] == P["entrenador"]["manager_id"]) & ~todos["prueba"], "torneo_corto"]))
                       & ~liga["prueba"], "match_id"]
tms = almacen.cargar("estadisticas_equipo", columnas=["match_id", "team_id", "team_match_sp_goals", "team_match_goals", *rivales.COLUMNAS])
medias, z = rivales.perfiles(tms[tms["match_id"].isin(ventana_ids)])
en_serie = tms[tms["match_id"].isin(serie["match_id"]) & (tms["team_id"] == rival_id)]
lig = medias.mean()
lecturas, iguales = [], 0
NOMBRES_RASGO = {"directness": "juego directo", "ppda": "PPDA (más alto = presiona menos)", "bp_favor": "goles esperados (xG) de balón parado a favor"}
for clave, nombre_rasgo in NOMBRES_RASGO.items():
    v, perfil = en_serie[rivales.RASGOS[clave][0]].mean(), medias.loc[rival_id, clave]
    igual = (v > lig[clave]) == (perfil > lig[clave])
    iguales += igual
    lecturas.append(f"{nombre_rasgo} {v:.2f} frente a {perfil:.2f} de su perfil (liga {lig[clave]:.2f}): "
                    + ("como marcaba el motor" if igual else "distinto a lo que marcaba el motor"))
goles_bp = int(en_serie["team_match_sp_goals"].sum())
goles_rival = int(serie["goles_contra"].sum())
ia = [ind.loc[mid][minuto.PROPIO].mean() for mid in serie["match_id"]]
caso = {
    "id": "eliminatoria", "titulo": f"Cuartos de final del CL26: América vs {nombre} ("
                                    + " y ".join(f"{int(a)}-{int(b)}" for a, b in zip(serie['goles_favor'], serie['goles_contra'])) + ")",
    "regla": "La eliminatoria del CL26: lo que decía nuestro motor del rival, con solo partidos anteriores.",
    "marcaba": (f"Lectura del motor sobre {nombre}: progresa — {', '.join(ficha['progresa'])}; defiende — {', '.join(ficha['defiende'])}; "
                f"concede — {', '.join(ficha['concede'])}; balón parado — {', '.join(ficha['balon_parado_rasgos']) or 'promedio'} "
                f"({ficha['balon_parado']})" if ficha else "Sin lectura del motor para este rival."),
    "paso": [f"En la serie, {nombre}: " + "; ".join(lecturas) + ".",
             f"El rival anotó {goles_rival} goles en los dos partidos; {goles_bp} de balón parado.",
             "Índice de iniciativa del América por partido: " + " y ".join(f"{v:.0f}" for v in ia) + "."],
    "lectura": (f"{nombre} jugó la serie {'como' if iguales == len(NOMBRES_RASGO) else 'distinto a'} su perfil en "
                f"{len(NOMBRES_RASGO) - iguales if iguales < len(NOMBRES_RASGO) else iguales} de {len(NOMBRES_RASGO)} rasgos"
                + (f"; aun así, {goles_bp} de sus {goles_rival} goles llegaron de balón parado, el peligro que marcaba el motor."
                   if ficha and "Peligroso a balón parado" in ficha["balon_parado_rasgos"] and goles_bp > 0 else ".")
                + " La lectura del motor describe al rival promedio; en una eliminatoria el rival se adapta."),
}
casos.append(caso)
(DIR_HALLAZGOS / "casos_cl26.json").write_text(json.dumps(casos, ensure_ascii=False, indent=1), encoding="utf-8")
caso

## Lectura

Corrida del 18 oct. Las reglas eligieron estos casos sin mirar los resultados.

1. **Ganando por 1 (América 2-1 Querétaro).**
    - Ganó por un gol durante 59 minutos (del 20' al 44' y del 61' al final).
    - No llegó al repliegue completo que marcaba la matriz: índice 54, frente a 57 de su partido normal.
    - El rival sí generó mucho más en esos minutos (1.17 xG contra 0.21): el partido se emparejó, como advertía la matriz.
2. **Cambio de sistema (América 0-1 Guadalajara, 4-2-3-1 → 3-4-3 al 12').**
    - Antes del cambio el rival dominaba (0.68 xG por 10 minutos).
    - Después bajó el xG del América, como anticipaba la alerta, pero el del rival bajó mucho más. El cambio frenó al rival.
    - La alerta describe el costo ofensivo del cambio, no si el cambio conviene.
3. **La eliminatoria (cuartos contra Pumas, 3-3 y 3-3).**
    - La lectura previa del motor marcaba un rival que va directo, presiona alto y es peligroso a balón parado.
    - En la serie, Pumas jugó distinto a su perfil en los tres rasgos medidos: elaboró más, presionó menos y generó menos a balón parado.
    - Aun así, 2 de sus 6 goles llegaron de balón parado.
    - El América propuso mucho (índice 61 y 79).
    - Lectura: el motor describe al rival promedio; en una eliminatoria el rival se adapta, y eso también es información para el cuerpo técnico.

En ningún caso se dice qué habría pasado con otra decisión. Se muestra qué marcaba la herramienta frente a lo que pasó.